# Generating sequences

To be able to train neural networks on sequential data, you need to pre-process it first. You'll chunk the data into inputs-target pairs, where the inputs are some number of consecutive data points and the target is the next data point.

In [1]:
import pandas as pd
train_data = pd.read_csv("dataset/electricity_train.csv")
train_data.head()

,timestamp,consumption
0,2011-01-01 00:15:00,-0.704319
1,2011-01-01 00:30:00,-0.704319
2,2011-01-01 00:45:00,-0.678983
3,2011-01-01 01:00:00,-0.653647
4,2011-01-01 01:15:00,-0.704319


In [2]:
import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    # Iterate over data indices
    for i in range(len(df) - seq_length):
      	# Define inputs
        x = df.iloc[i:(i+seq_length), 1]
        # Define target
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Sequential Dataset

Just like tabular and image data, sequential data is easiest passed to a model through a torch Dataset and DataLoader. To build a sequential Dataset, you will call create_sequences() to get the NumPy arrays with inputs and targets, and inspect their shape. Next, you will pass them to a TensorDataset to create a proper torch Dataset, and inspect its length

In [3]:
import torch
from torch.utils.data import TensorDataset

# Use create_sequences to create inputs and targets
X_train, y_train = create_sequences(train_data,96)
print(X_train.shape, y_train.shape)

# Create TensorDataset
dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),)
print(len(dataset_train))

(105119, 96) (105119,)
105119


# Sequential architectures

Whenever you face a task that requires handling sequential data, you need to be able to decide what type of recurrent architecture is the most suitable for the job. Let's test your understanding of when each architecture is applicable.

<center><img src="images/03.11.png"  style="width: 400px, height: 300px;"/></center>


<center><img src="images/03.12.png"  style="width: 400px, height: 300px;"/></center>


# Building a forecasting RNN

It's time to build your first recurrent network! It will be a sequence-to-vector model consisting of an RNN layer with two layers and a hidden_size of 32. After the RNN layer, a simple linear layer will map the outputs to a single value to be predicted.

In [4]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # Initialize first hidden state with zeros
        h0 = torch.zeros(2, x.size(0), 32)
        # Pass x and h0 through recurrent layer
        out, _ = self.rnn(x, h0)  
        # Pass recurrent layer's last output through linear layer
        out = self.fc(out[:, -1, :])
        return out

# RNN vs. LSTM vs. GRU

Understanding the differences between a plain RNN, LSTM, and GRU networks, including their complexity and computation needs, allows you to choose the appropriate one for your task.

Which of the following statements about the different types of recurrent neural networks are correct?

<center><img src="images/03.13.png"  style="width: 400px, height: 300px;"/></center>


- LSTM cells keep two hidden states: one for short-term memory and one for long-term memory.
- In both LSTM and GRU cells, the hidden state is not only passed as input to the next time step, but also returned at the current time step (that is, "y" and "h" are the same).

# LSTM network

As you already know, plain RNN cells are not used that much in practice. A more frequently used alternative that ensures a much better handling of long sequences are Long Short-Term Memory cells, or LSTMs. In this exercise, you will be build an LSTM network yourself!

The most important implementation difference from the RNN network you have built previously comes from the fact that LSTMs have two rather than one hidden states. This means you will need to initialize this additional hidden state and pass it to the LSTM cell.

In [5]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # Define lstm layer
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        # Initialize long-term memory
        c0 = torch.zeros(2, x.size(0), 32)
        # Pass all inputs to lstm layer
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# GRU network

Next to LSTMs, another popular recurrent neural network variant is the Gated Recurrent Unit, or GRU. It's appeal is in its simplicity: GRU cells require less computation than LSTM cells while often matching them in performance.

In [6]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.gru(x, h0)  
        out = self.fc(out[:, -1, :])
        return out

# RNN training loop

It's time to train the electricity consumption forecasting model!

You will use the LSTM network you have defined previously which is available to you as Net, as is the dataloader_train you built before. You will also need to use torch.nn which has already been imported as nn.

In [15]:
# from torch import optim
# from torch.utils.data import DataLoader
# net = Net()
# # Set up MSE loss
# criterion = nn.MSELoss()
# optimizer = optim.Adam(
#   net.parameters(), lr=0.0001
# )

# dataloader_train = DataLoader(dataset_train, batch_size=16)
# for epoch in range(1):
#     for seqs, labels in dataloader_train:
#       # print(seqs.shape)
#       # Reshape model inputs
#       seqs = seqs.view(16, 96, 1)
#       # print(seqs.shape)
#       # Get model outputs
#       outputs = net(seqs)
#       # Revert to original shape
#       outputs = outputs.squeeze()
#       # print(outputs.shape)
#       # print(labels.shape)
#         # Compute loss
#       loss = criterion(outputs, labels)
#       optimizer.zero_grad()
#       loss.backward()
#       optimizer.step()
#       # print(f"Epoch {epoch+1}, Loss: {loss.item()}")
# # loss = criterion.compute()

# Evaluating forecasting models

It's evaluation time! The same LSTM network that you have trained in the previous exercise has been trained for you for a few more epochs and is available as net.

In [16]:
# import torchmetrics
# # Define MSE metric
# dataloader_test = dataloader_train
# mse = torchmetrics.MeanSquaredError()

# net.eval()
# with torch.no_grad():
#     for seqs, labels in dataloader_test:
#         seqs = seqs.view(32, 96, 1)
#         # Pass seqs to net and squeeze the result
#         outputs = net(seqs).squeeze()
#         mse(outputs, labels)

# # Compute final metric value
# test_mse = mse.compute()
# print(f"Test MSE: {test_mse}")